## Example 1: wrapping a pandas DataFrame in the standard API

In [ ]:
import pandas as pd
import dataframe_api_compat
from dataframe_api_compat import pandas_standard

# Example 1: convert_to_standard_compliant_dataframe wraps a pandas DataFrame in the
# DataFrame API standard object; .dataframe gives the pandas object back.
print("Example 1: wrapping a pandas DataFrame")
assert set(dataframe_api_compat.__all__) >= {"pandas_standard", "polars_standard"}, dataframe_api_compat.__all__
pdf = pd.DataFrame({"a": [1, 2, 3, 4], "b": [10.0, 20.0, 30.0, 40.0]})
df = pandas_standard.convert_to_standard_compliant_dataframe(pdf)
assert isinstance(df, pandas_standard.DataFrame), type(df)
assert df.column_names == ["a", "b"], df.column_names
pd.testing.assert_frame_equal(df.dataframe, pdf)

ns = df.__dataframe_namespace__()
assert isinstance(ns, pandas_standard.Namespace), type(ns)
print(f"  dataframe_api_compat {dataframe_api_compat.__version__}, API version {ns.__dataframe_api_version__}")

# The standard is written for lazy libraries too, so shape() needs an explicit persist() first.
try:
    df.shape()
except ValueError as e:
    assert "persist" in str(e), e
    print("  shape() before persist() is refused:", str(e).splitlines()[0])
else:
    raise AssertionError("shape() worked without persist()")
assert df.persist().shape() == (4, 2), df.persist().shape()
print("Example 1 finished")

## Example 2: column expressions, assign and filter

In [ ]:
import pandas as pd
from dataframe_api_compat import pandas_standard

# Example 2: columns are expressions; assign adds them and filter keeps rows where a boolean column is true.
print("Example 2: column expressions, assign and filter")
pdf = pd.DataFrame({"a": [1, 2, 3, 4], "b": [10.0, 20.0, 30.0, 40.0]})
df = pandas_standard.convert_to_standard_compliant_dataframe(pdf)

doubled = df.assign((df.col("a") * 2).rename("a2"), (df.col("a") + df.col("b")).rename("total"))
result = doubled.dataframe
assert result.columns.tolist() == ["a", "b", "a2", "total"], result.columns.tolist()
assert result["a2"].tolist() == [2, 4, 6, 8] and result["total"].tolist() == [11.0, 22.0, 33.0, 44.0]

kept = df.filter(df.col("a") > 2).dataframe
expected = pdf[pdf["a"] > 2].reset_index(drop=True)
pd.testing.assert_frame_equal(kept.reset_index(drop=True), expected)
print(f"  assign added a2 and total; filter(a > 2) kept {len(kept)} rows, same as pandas")
print("Example 2 finished")

## Example 3: reductions, group_by and scalars

In [ ]:
import pandas as pd
from dataframe_api_compat import pandas_standard

# Example 3: reductions return a one-row DataFrame or a Scalar; a Scalar must be persisted before use in Python.
print("Example 3: reductions, group_by and scalars")
pdf = pd.DataFrame({"k": [1, 2, 1, 2], "a": [1, 2, 3, 4], "b": [10.0, 20.0, 30.0, 40.0]})
df = pandas_standard.convert_to_standard_compliant_dataframe(pdf)

sums = df.select("a", "b").sum().dataframe
assert sums.iloc[0].tolist() == [10.0, 100.0], sums

grouped = df.group_by("k").sum().dataframe.sort_values("k").reset_index(drop=True)
expected = pdf.groupby("k", as_index=False).sum()
assert grouped.to_dict("list") == expected.to_dict("list"), (grouped, expected)

mean_b = df.col("b").mean()
try:
    float(mean_b)
except RuntimeError as e:
    print("  float() on an unpersisted Scalar is refused:", str(e).splitlines()[0])
else:
    raise AssertionError("float() worked on an unpersisted Scalar")
assert float(mean_b.persist()) == 25.0
print(f"  sums {sums.iloc[0].tolist()}, group sums {grouped.to_dict('list')}, mean(b) = 25.0")
print("Example 3 finished")

## Example 4: sort, rename, drop, cast and schema

In [ ]:
import pandas as pd
from dataframe_api_compat import pandas_standard

# Example 4: reshaping operations and dtypes expressed with the namespace's dtype classes.
print("Example 4: sort, rename, drop, cast and schema")
pdf = pd.DataFrame({"a": [3, 1, 4, 2], "b": [30.0, 10.0, 40.0, 20.0], "c": [0, 0, 1, 1]})
df = pandas_standard.convert_to_standard_compliant_dataframe(pdf)
ns = df.__dataframe_namespace__()

ordered = df.sort("a", ascending=False).dataframe
assert ordered["a"].tolist() == [4, 3, 2, 1], ordered
assert df.rename({"a": "alpha"}).column_names == ["alpha", "b", "c"]
assert df.drop("c").column_names == ["a", "b"]

schema = df.schema
assert isinstance(schema["a"], ns.Int64) and isinstance(schema["b"], ns.Float64), schema
cast = df.cast({"a": ns.Float64()})
assert isinstance(cast.schema["a"], ns.Float64) and str(cast.dataframe["a"].dtype) == "float64"
print(f"  schema {{a: {type(schema['a']).__name__}, b: {type(schema['b']).__name__}}}; cast a -> {type(cast.schema['a']).__name__}")
print("Example 4 finished")

## Example 5: joins and missing values

In [ ]:
import numpy as np
import pandas as pd
from dataframe_api_compat import pandas_standard

# Example 5: a left join on a key column, and the null helpers on a column with a missing value.
print("Example 5: joins and missing values")
left = pandas_standard.convert_to_standard_compliant_dataframe(pd.DataFrame({"key": [1, 2, 1, 3], "v": [1.0, 2.0, 3.0, 4.0]}))
right = pandas_standard.convert_to_standard_compliant_dataframe(pd.DataFrame({"key": [1, 2], "w": [100, 200]}))
joined = left.join(right, how="left", left_on="key", right_on="key").dataframe
assert joined.columns.tolist() == ["key", "v", "w"], joined.columns.tolist()
assert joined["w"].tolist()[:3] == [100, 200, 100] and np.isnan(joined["w"].iloc[3]), joined
print(f"  left join: {len(joined)} rows, key 3 has no match")

col = pandas_standard.convert_to_standard_compliant_column(pd.Series([1.0, None, 3.0], name="c"))
assert col.is_null().column.tolist() == [False, True, False]
assert col.fill_null(0.0).column.tolist() == [1.0, 0.0, 3.0]
with_gap = pandas_standard.convert_to_standard_compliant_dataframe(pd.DataFrame({"c": [1.0, None, 3.0], "d": [1, 2, 3]}))
assert with_gap.drop_nulls().dataframe["d"].tolist() == [1, 3]
print("  is_null / fill_null on a column and drop_nulls on a DataFrame behave as expected")
print("Example 5 finished")